In [ ]:
import pyvista as pv
import numpy as np
import xml.etree.ElementTree as ET
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D
#from matplotlib.legend_handler import HandlerTuple
from matplotlib.legend_handler import HandlerBase
import matplotlib.patches as mpatches
from scipy.interpolate import griddata
import os
from scipy.spatial import Delaunay
import csv



class HandlerOverlay(HandlerBase):
    def __init__(self, gray_line_props, color_line_props):
        super().__init__()
        self.gray_props = gray_line_props
        self.color_props = color_line_props
    
    def create_artists(self, legend, orig_handle, xdescent, ydescent, 
                      width, height, fontsize, trans):
        # Create the gray line first (background)
        gray_line = Line2D([xdescent, xdescent + width], 
                          [ydescent + height/2, ydescent + height/2],
                          transform=trans, **self.gray_props)
        
        # Create the colored line on top
        color_line = Line2D([xdescent, xdescent + width], 
                           [ydescent + height/2, ydescent + height/2],
                           transform=trans, **self.color_props)
        
        return [gray_line, color_line]

def create_overlay_legend_entry(gray_props, color_props, label):
    """
    Create a single overlaid legend entry.
    
    Parameters:
    - gray_props: dict with properties for gray background line
    - color_props: dict with properties for colored foreground line
    - label: string label for legend
    
    Returns:
    - handle: Line2D handle for legend
    - handler: HandlerOverlay instance
    - label: the label string
    """
    handle = Line2D([], [])
    handler = HandlerOverlay(gray_props, color_props)
    return handle, handler, label

def get_fascicle_config():
    """Return fascicle configuration with properties and gray background settings."""
    fascicle_props = {
        "HB": {"color": "#FFA64D", "linestyle": "-"},
        "LB": {"color": "#FF4DFF", "linestyle": "-", "linewidth": 1.5},
        "RB": {"color": "#BD00BD", "linestyle": ":"},        
        "LV_fascicles": {"color": "#ff2994", "linestyle": ":"},        
        "LV153": {"color": "#00664f", "name": "LV sept branch", "marker": "D", "linewidth": 0.5}, 
        "LV154": {"color": "#003d7a", "name": "LV post branch", "marker": "o", "linewidth": 0.5},
        "LV155": {"color": "#cc6600", "name": "LV ant branch", "marker": "^", "linewidth": 0.5},  
        "RV_fascicles": {"color": "#800080", "linestyle": ":"},
        "TV_ring": {"color": "#fff32c", "linestyle": "--", "name": "20mm under \n TV annulus"}
        #"TV_ring": {"color": "#ce674d", "linestyle": "--", "name": "20mm under \n TV annulus"}
    }
    
    gray_props = {
        'color': "#BDBCBC",
        'linewidth': 4,
        'alpha': 0.5,
        'linestyle': '-'
    }
    
    return fascicle_props, gray_props
def get_AHA_config(): 
    AHA_mapping_regions = {
        2: {'theta_range': [270, 330], 'z_range': [0.67, 1]},
        3: {'theta_range': [210, 270], 'z_range': [0.67, 1]},
        8: {'theta_range': [270, 330], 'z_range': [0.33, 0.67]},
        9: {'theta_range': [210, 270], 'z_range': [0.33, 0.67]},
        14: {'theta_range': [225, 315], 'z_range': [0.0, 0.33]},
        20: {'theta_range': [260, 280], 'z_range': [0.45, 0.89], 'label': 'LBB', 'color': "#FF86FF", 'txtcol': "#FF4DFF"}, # LB 
        21: {'theta_range': [270, 330], 'z_range': [0.5, 0.75], 'label': 'LAF', 'color': "#fdb56c", 'txtcol': '#cc6600'}, # LV anterior branch area
        22: {'theta_range': [225, 270], 'z_range': [0.38, 0.625], 'label': 'LPF', 'color': '#4da6ff', 'txtcol': "#003d7a"}, # LV posterior branch area 
        23: {'theta_range': [252, 274], 'z_range': [0.25, 0.6], 'label': 'LSF', 'color': '#66b38c', 'txtcol': "#00664f"} # LV posterior branch area 
    }  

    
        #21: {'theta_range': [270, 330], 'z_range': [0.5, 0.75], 'label': 'Anterior Bundle', 'color': '#D8BFD8', 'txtcol': '#7D00B8'}, # LV anterior branch area
        #22: {'theta_range': [225, 270], 'z_range': [0.38, 0.625], 'label': 'Posterior Bundle', 'color': '#87CEFA', 'txtcol': "#0000FF"}, # LV posterior branch area 
        #23: {'theta_range': [252, 274], 'z_range': [0.25, 0.6], 'label': 'Septal Bundle', 'color': '#90EE90', 'txtcol': "#348B34"} # LV posterior branch area 
    return AHA_mapping_regions
def create_fascicle_dict(points_data, fascicle_props):
    """
    Create fascicle dictionary by combining point data with properties.
    
    Args:
        points_data: Dict mapping fascicle names to their point data
        fascicle_props: Dict with color and linestyle properties
        
    Returns:
        Dict with fascicle data including points, colors, and linestyles
    """
    # Mapping of fascicle names to their property keys and defaults
    fascicle_config = {
        "HB": {"prop_key": "HB", "default_linestyle": "-"},
        "LB": {"prop_key": "LB", "default_linestyle": "-"},
        "RB": {"prop_key": "RB", "default_linestyle": ":"},
        "LV_fascicles": {"prop_key": "LV_fascicles", "default_linestyle": "-"},
        "LV155": {"prop_key": "LV155", "default_linestyle": ":", "linestyle_source": "LV_fascicles"},
        "LV153": {"prop_key": "LV153", "default_linestyle": ":", "linestyle_source": "LV_fascicles"},
        "LV154": {"prop_key": "LV154", "default_linestyle": ":", "linestyle_source": "LV_fascicles"},
        "RV_fascicles": {"prop_key": "RV_fascicles", "default_linestyle": ":"}, 
        "TV_ring": {"prop_key": "TV_ring", "default_linestyle": "-"}, 
    }
    
    all_fascicles = {}
    
    for fascicle_name, points in points_data.items():
        if fascicle_name not in fascicle_config:
            continue
            
        config = fascicle_config[fascicle_name]
        prop_key = config["prop_key"]
        
        # Get color from properties
        color = fascicle_props[prop_key]["color"]
        
        # Get linestyle - check source if specified, otherwise use the prop_key
        linestyle_source = config.get("linestyle_source", prop_key)
        linestyle = fascicle_props[linestyle_source].get(
            "linestyle", 
            config["default_linestyle"]
        )
        
        # Get marker 
        marker=fascicle_props[prop_key].get("marker", "None")
        
        # Get linewidth
        linewidth=fascicle_props[prop_key].get("linewidth", 1)
        # Build the fascicle entry
        fascicle_entry = {
            "points": points,
            "color": color,
            "linestyle": linestyle,
            "marker": marker,
            "linewidth": linewidth
        }
        
        # Add name if it exists in the properties
        if "name" in fascicle_props[prop_key]:
            fascicle_entry["name"] = fascicle_props[prop_key]["name"]
            
        all_fascicles[fascicle_name] = fascicle_entry
    
    return all_fascicles

def set_all_fascicles(fascicle_props, all_HB_points, all_LB_points, all_RB_points,
                     all_LV_fascicles_points, all_RV_fascicles_points, all_LV153, 
                     all_LV154, all_LV155, all_TV_ring):
    """
    Create fascicle dictionary - wrapper for backward compatibility.
    
    Returns:
        Dict combining points with color/linestyle properties
    """
    points_data = {
        "HB": all_HB_points,
        "LB": all_LB_points,
        "RB": all_RB_points,
        "LV_fascicles": all_LV_fascicles_points,
        "LV155": all_LV155,
        "LV153": all_LV153,
        "LV154": all_LV154,
        "RV_fascicles": all_RV_fascicles_points,
        "TV_ring": all_TV_ring
    }
    
    return create_fascicle_dict(points_data, fascicle_props)


def add_CS_legend(all_fascicles=None, labels=None):
    """
    Add overlaid legend entries for CS data.
    
    Parameters:
    - all_fascicles: dict with fascicle data (contains color, linestyle, and optional name info)
    - labels: list of fascicle keys to create legend entries for. If None, uses all fascicles with data.
    """
    print('CS so make CS legend')
    
    # Get fascicle configuration if not provided
    if all_fascicles is None:
        fascicle_props, gray_base_props = get_fascicle_config()  # Using your refactored function
        # You'd need to call your set_all_fascicles function here with actual data
        # For now, just get the props
        all_fascicles = fascicle_props
    else:
        _, gray_base_props = get_fascicle_config()
    
    # If no labels specified, determine which fascicles actually have data to plot
    if labels is None:
        labels = []
        for name, fascicle_data in all_fascicles.items():
            # Check if this fascicle has any non-empty point data
            if "points" in fascicle_data:
                if any(np.asarray(fascicle).size > 0 for fascicle in fascicle_data["points"]):
                    labels.append(name)
            else:
                # If no points data, include it (for backward compatibility)
                labels.append(name)
    
    # Color overlay properties (consistent linewidth)
    color_base_props = {
        'linewidth': 1.5,
        'alpha': 1
    }
    
    # Default styles as fallback
    default_styles = {
        "LB": {"color": "#d63384", "linestyle": "-"},
        "RB": {"color": "#800080", "linestyle": ":"},
        "HB": {"color": "#555555", "linestyle": "-"},
        "LV_fascicles": {"color": "#d63384", "linestyle": "-"},
        "RV_fascicles": {"color": "#800080", "linestyle": ":"}
    }
    
    # Create legend entries dynamically based on labels
    handles = []
    legend_labels = []
    handler_map = {}
    
    for label in labels:
        # Get style from all_fascicles or use defaults
        if label in all_fascicles:
            fascicle_data = all_fascicles[label]
            color = fascicle_data.get('color', default_styles.get(label, {}).get('color', '#000000'))
            linestyle = fascicle_data.get('linestyle', default_styles.get(label, {}).get('linestyle', '-'))
            marker = fascicle_data.get('marker', 'None')
            linewidth = fascicle_data.get('linewidth', 1)
            # Use custom name if available, otherwise use the key
            display_name = fascicle_data.get('name', label)
            
        else:
            # Use default styles
            style = default_styles.get(label, {"color": "#000000", "linestyle": "-"})
            color = style['color']
            linestyle = style['linestyle']
            display_name = label
            marker='None'
            linewidth=1
        
        # Create gray properties with linestyle
        gray_props = gray_base_props.copy()
        # Uncomment if you want gray background to match linestyle:
        # gray_props['linestyle'] = linestyle
        
        # Create color properties with linestyle
        color_props = color_base_props.copy()
        color_props['color'] = color
        color_props['linestyle'] = linestyle
        color_props['marker']=marker
        color_props['linewidth']=linewidth
        
        # Create overlay legend entry
        entry = create_overlay_legend_entry(gray_props, color_props, display_name)
        handles.append(entry[0])
        legend_labels.append(entry[2])
        handler_map[entry[0]] = entry[1]
    
    # Create the legend
    #plt.legend(handles, legend_labels, handler_map=handler_map, loc='upper right')
    plt.legend(handles, legend_labels, handler_map=handler_map, 
           bbox_to_anchor=(0.7, 1.05), loc='upper left')
    return handles, legend_labels, handler_map

def read_xml_point_file(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    # Find all <point> entries under the first <time_series>
    points = []
    for point in root.findall('.//point_set/time_series/point'):
        x = float(point.find('x').text)
        y = float(point.find('y').text)
        z = float(point.find('z').text)
        points.append([x, y, z])

    return np.array(points)

def normalize(v):
    return v / np.linalg.norm(v)

def convert_to_polar_2d(mesh_points, pts, scale_z=True, verbose=False):
    apex = np.array(pts[0])
    base = np.array(pts[1])
    rv0 = np.array(pts[2])
    rv1 = np.array(pts[3])

    z_axis = normalize(base - apex)
    relative_points = mesh_points - apex
    base_dist = np.linalg.norm(base - apex)

    z_components = np.dot(relative_points, z_axis)[:, np.newaxis] * z_axis
    radial_vectors = relative_points - z_components

    z_coords = np.dot(relative_points, z_axis)
    apex_z_location= np.dot(apex-apex, z_axis)
    base_z_location = np.dot(base-apex, z_axis)                         
                             
    if verbose: 
        print('Z coordinates range:', np.min(z_coords), np.max(z_coords))
        print('Apex z location:', apex_z_location, 'Base z location:', base_z_location)
    if scale_z:
        # Normalize z_coords to [0, 1]
        z_coords = (z_coords - np.min(z_coords)) / (np.max(z_coords) - np.min(z_coords))
    else: 
        # scale to apex base distance
        z_coords = z_coords / base_dist

    ref_vector = normalize(rv0 - apex - np.dot((rv0 - apex), z_axis) * z_axis)
    y_ref = normalize(np.cross(z_axis, ref_vector))
    x_ref = ref_vector

    x_comp = np.dot(radial_vectors, x_ref)
    y_comp = np.dot(radial_vectors, y_ref)
    theta = np.degrees(np.arctan2(y_comp, x_comp)) % 360
    if verbose:
        print('Theta range:', np.min(theta), np.max(theta))
    # Get rv0 and rv1 current angles
    rv0_vec = rv0 - apex - np.dot((rv0 - apex), z_axis) * z_axis
    rv1_vec = rv1 - apex - np.dot((rv1 - apex), z_axis) * z_axis

    rv0_x = np.dot(rv0_vec, x_ref)
    rv0_y = np.dot(rv0_vec, y_ref)
    rv1_x = np.dot(rv1_vec, x_ref)
    rv1_y = np.dot(rv1_vec, y_ref)

    rv0_theta = np.degrees(np.arctan2(rv0_y, rv0_x)) % 360
    rv1_theta = np.degrees(np.arctan2(rv1_y, rv1_x)) % 360

    # Compute angular separation
    delta_theta = (rv1_theta - rv0_theta) % 360
    scale_factor = 120 / delta_theta  # 330 - 210 = 120

    # Remap full circle
    theta_rel = (theta - rv0_theta) % 360
    # If theta_rel is >300, then subtract 360
    theta_rel[theta_rel> 300] -= 360
    theta_scaled = (theta_rel * scale_factor) % 360
    theta_final = (theta_scaled + 210) % 360
    if verbose:
        print('rv0 theta:', rv0_theta, 'rv1 theta:', rv1_theta)
        print('Min theta:', np.min(theta_rel), 'Max theta:', np.max(theta_rel))
        print(f"rv0 θ = {rv0_theta:.2f}° → 210°, rv1 θ = {rv1_theta:.2f}° → 330°")
        print(f"Angular separation = {delta_theta:.2f}°, scale = {scale_factor:.4f}")
        print(f"Theta range: {np.min(theta_final):.2f}° to {np.max(theta_final):.2f}°")

    polar_coords = np.vstack((theta_final, z_coords)).T
    return polar_coords

def extract_fascicles(final_network_path='final_network_flip.vtk', points_array=None):
    if points_array is None:
        raise ValueError("points_array must be provided to extract fascicles.")

    mesh = pv.read(final_network_path)
    if isinstance(mesh, pv.UnstructuredGrid):
        mesh = mesh.extract_surface()
    HB_indices = np.where(mesh.cell_data['elemTag'] == 150)[0]
    LB_indices = np.where(mesh.cell_data['elemTag'] == 151)[0]
    RB_indices = np.where(mesh.cell_data['elemTag'] == 152)[0]
    LV153_indices = np.where(mesh.cell_data['elemTag'] == 153)[0]
    LV154_indices = np.where(mesh.cell_data['elemTag'] == 154)[0]
    LV155_indices = np.where(mesh.cell_data['elemTag'] == 155)[0]
    LV_fascicles_indices = np.where(np.isin(mesh.cell_data['elemTag'], [153, 154, 155]))[0]
    RV_fascicles_indices = np.where(np.isin(mesh.cell_data['elemTag'], [156, 157]))[0]

    HB = mesh.extract_cells(HB_indices).points
    LB = mesh.extract_cells(LB_indices).points
    RB = mesh.extract_cells(RB_indices).points
    LV153 = mesh.extract_cells(LV153_indices).points
    LV154 = mesh.extract_cells(LV154_indices).points
    LV155 = mesh.extract_cells(LV155_indices).points
    LV_fascicles = mesh.extract_cells(LV_fascicles_indices).points
    RV_fascicles = mesh.extract_cells(RV_fascicles_indices).points
    
    HB_points = convert_to_polar_2d(HB, points_array, scale_z=False)
    LB_points = convert_to_polar_2d(LB, points_array, scale_z=False)
    RB_points = convert_to_polar_2d(RB, points_array, scale_z=False)
    LV153_points = convert_to_polar_2d(LV153, points_array, scale_z=False)
    LV154_points = convert_to_polar_2d(LV154, points_array, scale_z=False)
    LV155_points = convert_to_polar_2d(LV155, points_array, scale_z=False)   
    
    #print('153')
    #print(f'MIN: {np.min(LB_points[:,0]):1.0f} MAX: {np.max(LB_points[:,0]):1.0f} || MIN: {np.min(LB_points[:,1]):1.3f} MAX: {np.max(LB_points[:,1]):1.3f}')
    #print(f'MIN: {np.min(LV153_points[:,0]):1.0f} MAX: {np.max(LV153_points[:,0]):1.0f} || MIN: {np.min(LV153_points[:,1]):1.3f} MAX: {np.max(LV153_points[:,1]):1.3f}')

    #print(np.max(LV155_points[:,1]))
    # Apply threshold: keep only rows where column 0 is < 330
    #mask = LV155_points[:,0] < 330
    #filtered_points = LV155_points[mask]

    # Find the max of column 1 in the filtered data
    #max_after_threshold = np.max(filtered_points[:,1])
    
    LV_fascicles_points = convert_to_polar_2d(LV_fascicles, points_array, scale_z=False)
    RV_fascicles_points = convert_to_polar_2d(RV_fascicles, points_array, scale_z=False)   

    return HB_points, LB_points, RB_points, LV_fascicles_points, RV_fascicles_points, LV153_points, LV154_points, LV155_points

def extract_TV_ring(TV_ringfile='TV_ring-0.vtk', points_array=None): 
    mesh=pv.read(TV_ringfile)
    if isinstance(mesh, pv.UnstructuredGrid):
        mesh = mesh.extract_surface()
    TV_ring=mesh.points
    TV_points=convert_to_polar_2d(TV_ring, points_array, scale_z=False)
    return TV_points

def split_mesh_by_elemTag(mesh, tag_name='elemTag', onpts=True,verbose=False):
    """
    Splits a PyVista mesh into RV free wall and septum based on an element tag.

    Parameters:
    - mesh: pyvista.UnstructuredGrid or PolyData
    - tag_name: name of the cell array to use for splitting (default 'elemTag')

    Returns:
    - rv_freewall_mesh: mesh with elemTag == 0
    - septum_mesh: mesh with elemTag == 3
    """    
    
    if tag_name not in mesh.cell_data:
        raise ValueError(f"Tag '{tag_name}' not found in mesh cell data.")
    
    if onpts: # If onpts is True, we will average the elemTag across points
        elem_tags = mesh.cell_data["elemTag"]
        # print out the unique elem_tags and their counts
        unique, counts = np.unique(elem_tags, return_counts=True)
        #if verbose: 
        print(f'Element tags and counts: {dict(zip(unique, counts))}')
        point_tags = [[] for _ in range(mesh.n_points)]

        for cell_id in range(mesh.n_cells):
            for pt_id in mesh.get_cell(cell_id).point_ids:
                point_tags[pt_id].append(elem_tags[cell_id])

        # Average or first
        avg_point_tags = np.array([np.mean(t) if t else -1 for t in point_tags], dtype=int)
        if verbose: 
            print(f'Average point tags number of points: {avg_point_tags.shape[0]}')
        mesh.point_data["elemTag"] = avg_point_tags
        
        rv_freewall_indices = np.where(avg_point_tags == 0)[0]
        septum_indices = np.where(avg_point_tags == 3)[0]
        if verbose: 
            print(f'RV Free Wall number of points: {rv_freewall_indices.shape[0]}')
            print(f'Septum number of points: {septum_indices.shape[0]}')

        # Remove overlapping indices from septum_indices
        septum_indices = np.setdiff1d(septum_indices, rv_freewall_indices, assume_unique=False)
        if verbose: 
            print(f'Septum number of points after removing RV Free Wall: {septum_indices.shape[0]}')

    else: 
        #rv_freewall_mesh = mesh.threshold(value=0.1, scalars=tag_name, invert=True)  # elemTag == 0
        #septum_mesh = mesh.threshold(value=(2.9, 3.1), scalars=tag_name)            # elemTag == 3
        
        rv_freewall_indices = np.where(mesh.cell_data[tag_name] == 0)[0]
        septum_indices = np.where(mesh.cell_data[tag_name] == 3)[0] 
        # rv_freewall_mesh = mesh.extract_cells(rv_freewall_indices)
        # septum_mesh = mesh.extract_cells(septum_indices)

    return rv_freewall_indices, septum_indices

def plot_mesh_pts(fig, ax, polar_points, field, lm_points, title='RV Free Wall in Polar Coordinates'):
# plot rv_points and septum_points in polar coordinates in a cicular plot
    
    ax.set_theta_zero_location('N')  # Set 0 degrees at the top
    ax.set_theta_direction(-1)  # Set direction to clockwise                
    ax.set_ylim(0, 1)  # Set radial limits to [0, 1]
    ax.scatter(np.radians(polar_points[:, 0]), polar_points[:, 1], c=field, cmap='viridis', s=10)

    # Separate the data based on field values
    no_collision_mask = field == 0
    collision_mask = field == 1

    # Plot each category separately with explicit labels
    ax.scatter(np.radians(polar_points[no_collision_mask, 0]), 
            polar_points[no_collision_mask, 1], 
            c='blue', label='No collision', s=10)
    ax.scatter(np.radians(polar_points[collision_mask, 0]), 
            polar_points[collision_mask, 1], 
            c='red', label='Collision', s=10)
    #ax.scatter(np.radians(rv_points[:, 0]), rv_points[:, 1], color='blue', label='RV Free Wall', s=10)
    #ax.scatter(np.radians(septum_points[:, 0]), septum_points[:, 1], color='red', label='Septum', s=10)
    ax.set_title(title, fontsize=14, pad=20)
    ax.legend()

    # Change axis labels to be in degrees 30, 90, 120, 150, 180, 210, 240, 270, 300, 330
    ax.set_xticks(np.radians([30, 90, 150, 210, 270, 330]))
    ax.set_xticklabels(['', '', '', '', '', ''])  
    ax.text(np.radians(90), 1.05, 'Lateral', ha='center', va='center', fontsize=10, color='black', rotation=-90)
    ax.text(np.radians(270), 1.05, 'Septal', ha='center', va='center', fontsize=10, color='black', rotation=90)
    ax.text(np.radians(0), 1.05, 'Anterior', ha='center', va='center', fontsize=10, color='black')
    ax.text(np.radians(180), 1.05, 'Inferior', ha='center', va='center', fontsize=10, color='black')
    ax.set_yticks([0.3333, 0.6666, 1])  # Hide radial ticks
    ax.set_yticklabels(['', '', '']) 

    #ax.scatter(np.radians(lm_points[:, 0]), lm_points[:, 1], color='green', label='XML Points', s=10)

    # Add labels for the points
    RVcusp_label=['Apex', 'Base', 'RV cusp inf (TV)', 'RV cusp ant (RVOT)']
    for i, (theta, z) in enumerate(lm_points):
        if i > 1: # Skip the first two points (Apex and Base)
            ax.text(np.radians(theta), z, RVcusp_label[i], fontsize=8, ha='center', va='center')
    #plt.show()
    return fig, ax

def interpolate_field(polar_points, field):

    theta_all = polar_points[:, 0]
    z_all = polar_points[:, 1]
    field_all = field
    
    # Create regular grid in polar coordinates
    theta_grid = np.linspace(210, 330, 180)
    z_grid = np.linspace(0, 1, 200)
    theta_grid_rad = np.radians(theta_grid)
    Theta, Z = np.meshgrid(theta_grid_rad, z_grid)
    
    # Interpolate the field onto the grid
    interp_field = griddata(
        points=(np.radians(theta_all), z_all),
        values=field_all,
        xi=(Theta, Z),
        method='nearest'
    )
    
    #out_polar_points = np.vstack((theta_grid, z_grid)).T
    # Return the interpolated field and the grid
    return Theta, Z, interp_field

def extract_region_from_grid(interp_field, theta_grid_deg, z_grid, theta_range, z_range):
    # Find index ranges corresponding to theta and z
    theta_mask = (theta_grid_deg >= theta_range[0]) & (theta_grid_deg <= theta_range[1])
    z_mask = (z_grid >= z_range[0]) & (z_grid <= z_range[1])
    
    # Use np.ix_ to extract the region
    region_data = interp_field[np.ix_(z_mask, theta_mask)]
    return region_data

def plot_mesh_interpolate(fig, ax, polar_points, field, lm_points, title='RV Free Wall in Polar Coordinates'):

    # Combine points and field
    theta_all = polar_points[:, 0]
    z_all = polar_points[:, 1]
    field_all = field

    # Create regular grid in polar coordinates
    theta_grid = np.linspace(210, 330, 180)
    z_grid = np.linspace(0, 1, 200)
    theta_grid_rad = np.radians(theta_grid)
    Theta, Z = np.meshgrid(theta_grid_rad, z_grid)

    # Interpolate the field onto the grid
    interp_field = griddata(
        points=(np.radians(theta_all), z_all),
        values=field_all,
        xi=(Theta, Z),
        method='nearest'
    )

    # Plot the interpolated surface
    # fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 1)

    # Use pcolormesh to plot the interpolated field
#    c = ax.pcolormesh(Theta, Z, interp_field, cmap='viridis', shading='auto')
    # Define binary colormap: 0 = blue, 1 = red
    binary_cmap = ListedColormap(['blue', 'red'])

    # Round interpolated field to binary (if not already 0/1)
    binary_field = np.round(interp_field)

    # Plot using custom binary colormap
    c = ax.pcolormesh(Theta, Z, binary_field, cmap=binary_cmap, shading='auto', vmin=0, vmax=1)
    fig.colorbar(c, ax=ax, ticks=[0, 1], label='Collision (0 = No, 1 = Yes)')

    #fig.colorbar(c, ax=ax, label='Collision Value')

    #ax.scatter(np.radians(rv_points[:, 0]), rv_points[:, 1], color='blue', label='RV Free Wall', s=10)
    #ax.scatter(np.radians(septum_points[:, 0]), septum_points[:, 1], color='red', label='Septum', s=10)
    ax.set_title(title, fontsize=14, pad=20)
    ax.legend()

    # Change axis labels to be in degrees 30, 90, 120, 150, 180, 210, 240, 270, 300, 330
    ax.set_xticks(np.radians([30, 90, 150, 210, 270, 330]))
    ax.set_xticklabels(['', '', '', '', '', ''])  
    ax.text(np.radians(90), 1.05, 'Lateral', ha='center', va='center', fontsize=10, color='black', rotation=-90)
    ax.text(np.radians(270), 1.05, 'Septal', ha='center', va='center', fontsize=10, color='black', rotation=90)
    ax.text(np.radians(0), 1.05, 'Anterior', ha='center', va='center', fontsize=10, color='black')
    ax.text(np.radians(180), 1.05, 'Inferior', ha='center', va='center', fontsize=10, color='black')
    ax.set_yticks([0.3333, 0.6666, 1])  # Hide radial ticks
    ax.set_yticklabels(['', '', '']) 

    #ax.scatter(np.radians(lm_points[:, 0]), lm_points[:, 1], color='green', label='XML Points', s=10)

    # Add labels for the points
    RVcusp_label=['Apex', 'Base', 'RV cusp inf (TV)', 'RV cusp ant (RVOT)']
    for i, (theta, z) in enumerate(lm_points):
        if i > 1: # Skip the first two points (Apex and Base)
            ax.text(np.radians(theta), z, RVcusp_label[i], fontsize=8, ha='center', va='center')
    #plt.show()
    return fig, ax
def add_region_wedge(ax): 
    AHA_mapping_regions = get_AHA_config()
    #AHA_mapping_regions = {
    #    20: {'theta_range': [270, 330], 'z_range': [0.5, 0.735], 'label': 'LV ant branch'}, # LV anterior branch area 
    #    21: {'theta_range': [225, 270], 'z_range': [0.38, 0.625], 'label': 'LV post branch'} # LV posterior branch area 
    #}

    #plot_fascicles(ax, all_fascicles)
    # Method 1: Using filled patches (recommended)
    #colors = ['lightblue', 'lightcoral',]

    for i, (region_id, region_data) in enumerate(AHA_mapping_regions.items()):
        if region_id<20: 
            continue
        theta_start, theta_end = region_data['theta_range']
        r_start, r_end = region_data['z_range']
        color=region_data['color']
        
        # Convert degrees to radians
        theta_start_rad = np.radians(theta_start)
        theta_end_rad = np.radians(theta_end)
        

        # Create and add the patch using fill_between for proper curves
        theta_region = np.linspace(theta_start_rad, theta_end_rad, 100)
        ax.fill_between(theta_region, r_start, r_end, 
                    alpha=0.3, color=color)
        
    # Method 2: Using curved outline only (alternative approach)
    # Uncomment below if you prefer just curved outlines:

    for region_id, region_data in AHA_mapping_regions.items():
        if region_id<20: 
            continue

        theta_start, theta_end = region_data['theta_range']
        r_start, r_end = region_data['z_range']
        
        # Convert to radians
        theta_start_rad = np.radians(theta_start)
        theta_end_rad = np.radians(theta_end)
        
        # Create smooth curves
        n_points = 50
        theta_arc = np.linspace(theta_start_rad, theta_end_rad, n_points)
        
        # Draw curved boundaries
        # Inner arc
        ax.plot(theta_arc, [r_start] * len(theta_arc), ':', linewidth=1, color="#000000")
        # Outer arc
        ax.plot(theta_arc, [r_end] * len(theta_arc), 'k:', linewidth=1)
        # Left radial line
        ax.plot([theta_start_rad, theta_start_rad], [r_start, r_end], 'k-', linewidth=1)
        # Right radial line
        ax.plot([theta_end_rad, theta_end_rad], [r_start, r_end], 'k-', linewidth=1)


    # Add labels and formatting
    #ax.set_title('AHA Mapping Regions on Polar Plot')
    ax.grid(True, alpha=0.3)

    # Optional: Add region labels at center of each region
    for region_id, region_data in AHA_mapping_regions.items():
        if region_id<20: 
            continue
        theta_start, theta_end = region_data['theta_range']
        r_start, r_end = region_data['z_range']
        label=region_data['label']
        txtcol=region_data['txtcol']
        # Calculate center position
        theta_center = np.radians(((theta_start + theta_end) / 2)) 
        r_center = (r_start + r_end) / 2
        
        if region_id == 23: 
            r_center=0.3
            theta_center=np.radians(285)
        if region_id==20: 
            r_center=0.8
        ax.annotate(label, color=txtcol, fontsize=12,
                xy=(theta_center, r_center), 
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    
    return ax
        
def add_AHA_lines(ax, full_circle):
    # Basal level (segments 1-6) - 6 segments of 60° each
    basal_angles = np.array([30, 90, 150, 210, 270, 330]) * np.pi / 180

    # Mid-cavity level (segments 7-12) - 6 segments of 60° each  
    mid_angles = np.array([30, 90, 150, 210, 270, 330]) * np.pi / 180

    # Apical level (segments 13-16) - 4 segments of 90° each
    apical_angles = np.array([45, 135, 225, 315]) * np.pi / 180

    # Radial boundaries (from center to edge)
    # Assuming normalized radial coordinates: 0 = center, 1 = edge
    apical_radius = 0.33   # Inner third (apex)
    mid_radius = 0.67      # Middle third  
    basal_radius = 1.0     # Outer third (base)

    # Add the segment boundary lines to your existing polar plot
    # Replace your current axis labeling code with this:

    # Clear any existing tick labels
    ax.set_xticks([])
    ax.set_xticklabels([])
    ax.set_yticks([])
    ax.set_yticklabels([])

    # Draw radial boundary lines (concentric circles)
    circle1 = plt.Circle((0, 0), apical_radius, fill=False, color='black', linewidth=1, transform=ax.transData._b)
    circle2 = plt.Circle((0, 0), mid_radius, fill=False, color='black', linewidth=1, transform=ax.transData._b)
    ax.add_patch(circle1)
    ax.add_patch(circle2)

    # Draw angular boundary lines for basal segments (1-6)
    for angle in basal_angles[:]:  # Exclude 360° as it's same as 0°
        ax.plot([angle, angle], [mid_radius, basal_radius], 'k-', linewidth=1)

    # Draw angular boundary lines for mid-cavity segments (7-12)  
    for angle in mid_angles[:]:  # Exclude 360° as it's same as 0°
        ax.plot([angle, angle], [apical_radius, mid_radius], 'k-', linewidth=1)

    # Draw angular boundary lines for apical segments (13-16)
    for angle in apical_angles[:]:  # Exclude 360° as it's same as 0°
        ax.plot([angle, angle], [0, apical_radius], 'k-', linewidth=1)

    # Optional: Add segment labels at the center of each segment
    segment_labels = {
        # Basal segments (1-6)
        1: (0 * np.pi/180, (mid_radius + basal_radius)/2),   # Basal anterior
        2: (300 * np.pi/180, (mid_radius + basal_radius)/2),   # Basal anteroseptal  
        3: (240 * np.pi/180, (mid_radius + basal_radius)/2),  # Basal inferoseptal
        4: (180 * np.pi/180, (mid_radius + basal_radius)/2),  # Basal inferior
        5: (120 * np.pi/180, (mid_radius + basal_radius)/2),  # Basal inferolateral
        6: (60 * np.pi/180, (mid_radius + basal_radius)/2),  # Basal anterolateral
        
        # Mid-cavity segments (7-12)
        7: (0 * np.pi/180, (apical_radius + mid_radius)/2),   # Mid anterior
        8: (300 * np.pi/180, (apical_radius + mid_radius)/2),   # Mid anteroseptal
        9: (240 * np.pi/180, (apical_radius + mid_radius)/2),  # Mid inferoseptal  
        10: (180 * np.pi/180, (apical_radius + mid_radius)/2), # Mid inferior
        11: (120 * np.pi/180, (apical_radius + mid_radius)/2), # Mid inferolateral
        12: (60 * np.pi/180, (apical_radius + mid_radius)/2), # Mid anterolateral
        
        # Apical segments (13-16)
        13: (0 * np.pi/180, apical_radius/2),   # Apical anterior
        14: (270 * np.pi/180, apical_radius/2),  # Apical septal
        15: (180 * np.pi/180, apical_radius/2),  # Apical inferior  
        16: (90 * np.pi/180, apical_radius/2),  # Apical lateral
    }
    #full_circle=False
    if full_circle: 
        # Add segment numbers (optional)
        for segment_num, (angle, radius) in segment_labels.items():
            ax.text(angle, radius, str(segment_num), ha='center', va='center', 
                    fontsize=8, fontweight='bold', color='black')

        # Add anatomical direction labels at the perimeter
        ax.text(np.radians(90), 1.05, 'Lateral', ha='center', va='center', fontsize=10, fontweight='bold', color='black', rotation=-90)
        ax.text(np.radians(0), 1.05, 'Anterior', ha='center', va='center', fontsize=10, fontweight='bold', color='black')
        ax.text(np.radians(180), 1.05, 'Inferior', ha='center', va='center', fontsize=10, fontweight='bold', color='black')

    else: 
        segments_to_show = [2, 3, 8, 9, 14]
        for segment_num, (angle, radius) in segment_labels.items():
            if segment_num in segments_to_show:
                ax.text(angle, radius, str(segment_num), ha='center', va='center',
                        fontsize=8, fontweight='bold', color='black')
        #ax.text(331 * np.pi/180, 0.825, 'RVOT' , ha='center', va='center',
        #                fontsize=8, fontweight='bold', color='black', rotation=-60)
        #ax.text(240 * np.pi/180, 1.05, 'TV annulus' , ha='center', va='center',
        #                fontsize=8, fontweight='bold', color='black', rotation=120)
                
    # Always show septal label
    ax.text(np.radians(270), 1.05, 'Septal', ha='center', va='center', fontsize=10, fontweight='bold', color='black', rotation=90)
def get_diameter(device, length): 
    devices={
        "Micra": {"D": 6.7, "L": 25.9}, # AV/AV2
        "AveirAR": {"D": 6.5, "L": 32.2}, # atrial Aveir DR device 
        "AveirVR": {"D": 6.5, "L": 38} # Atrial Aveir DR device + 10mm
    }

    D = devices[device]["D"]
    L = devices[device]["L"]
    
    # Calculate original volume
    vol = np.pi * ((D / 2) ** 2) * L
    # If we want to keep the same volume, calculate new diameter for each new length    
    new_D = np.sqrt(vol / (np.pi * length))*2 
    return new_D
    
def create_collision_colormap(collision_field, n_levels=10):
    """
    Create a colormap based on the collision field type.
    
    Parameters:
    collision_field (str): The type of collision field ('TV_Collision', 'RV_Collision', or 'PM_Collision')
    n_levels (int): Number of levels for the colormap (default: 10)
    
    Returns:
    tuple: (cmap, norm) - colormap and normalization objects
    """

    if collision_field.startswith('TV_Collision'):
        # White to blue gradient
        colors = [
            "#ffffff", "#b3d9ff", "#8cc6ff", "#66b3ff", "#4da6ff",
            "#3399ff", "#1a8cff", "#0073e6", "#005bb3", "#003d7a"
        ]
        under_color = "#4da6ff"
        contour_colour =  "#003d7a" #"#005080"
        #alphas=[1,1,1,1,1,1,1,1,1,1]
    elif collision_field.startswith('PM_Collision'):
        # White to green gradient
        colors = [
            "#ffffff","#cce6db", "#b3d9c7", "#99ccb3", "#80bf9f",
            "#66b38c", "#4da678", "#339964", "#1a8051", "#00664f"
        ]
        under_color = "#66b38c"
        contour_colour="#00664f"
        alphas=[0,0.2,0.3,0.4,0.5,1,1,1,1,1]

    elif collision_field.startswith('RV_Collision'):
        # White to orange gradient
        colors = [
            "#ffffff", "#ffcc99", "#ffb366", "#ff9933", "#ff8000", "#e6731a",
            "#cc6600", "#b35900", "#994d00", "#804000"
        ]
        
        under_color = "#ff9933"
        contour_colour = "#cc6600" # "#993d00"
        #alphas=[1,1,1,1,1,1,1,1,1,1]
    else:
        # Default: original blue-red diverging colormap
        colors = [
            "#2166ac", "#4393c3", "#92c5de", "#d1e5f0", "#f7f7f7",
            "#fddbc7", "#f4a582", "#d6604d", "#b2182b", "#67001f"
        ]
        under_color = "#2166ac"
        contour_colour="black"
    

    # Convert hex to RGB and add alpha
    def hex_to_rgba(hex_color, alpha):
        """Convert hex color to RGBA tuple"""
        hex_color = hex_color.lstrip('#')
        rgb = tuple(int(hex_color[i:i+2], 16)/255.0 for i in (0, 2, 4))
        return rgb + (alpha,)

    # Create alpha values from 0 (white) to 0.5 (orange)
    #alphas = np.linspace(0, 1, len(colors))
    #alphas=[0,0,0,0,0,0.8,0.8,.8,0.8,0.8]
    alphas=[0,1,1,1,1,1,1,1,1,1]
    # Convert to RGBA
    colors_rgba = [hex_to_rgba(color, alpha) for color, alpha in zip(colors, alphas)]

    # Create colormap
    cmap = ListedColormap(colors_rgba)
    # Create colormap
    #cmap = ListedColormap(colors)
    cmap.set_under(under_color)  # values < vmin will appear in the base color
    
    # Define scalar bounds for normalization (e.g., 0 to 100)
    bounds = np.linspace(0, 100, n_levels + 1)  # n_levels+1 edges for n_levels bins
    norm = BoundaryNorm(bounds, cmap.N)
    
    return cmap, bounds, norm, contour_colour, under_color

def plot_fascicles(ax, all_fascicles=None): 
    # Default to empty dict if None
    if all_fascicles is None:
        all_fascicles = {}
    
    for name, fascicle_data in all_fascicles.items():
        #grey_color=fascicle_data.get("grey_color", "#808080")  # default black
        _, gray_base_props = get_fascicle_config()

        for fascicle in fascicle_data["points"]:
            fascicle = np.asarray(fascicle)
            if fascicle.size == 0:
                continue
            theta = np.radians(fascicle[:, 0])
            r = fascicle[:, 1]
            ax.plot(theta, r, color=gray_base_props['color'], linewidth=gray_base_props['linewidth'], alpha=gray_base_props['alpha'])

    # Plot each fascicle set as its own colored line - only do once per plot 
    for name, fascicle_data in all_fascicles.items():
        color = fascicle_data.get("color", "#000000")  # default black
        linestyle = fascicle_data.get("linestyle", '-')
        # Use the 'name' field if it exists, otherwise use the key
        label = fascicle_data.get("name", name)
        marker = fascicle_data.get("marker", 'None')
        linewidth=fascicle_data.get("linewidth", 1)
         # Track if we've added the label for this fascicle set
        label_added = False

        for fascicle in fascicle_data["points"]:
            fascicle = np.asarray(fascicle)
            if fascicle.size == 0:
                continue
            # Get every nth point 
            n=5
            theta = np.radians(fascicle[::n, 0])
            r = fascicle[::n, 1]            


            # Only add label to the first non-empty fascicle in the set
            current_label = label if not label_added else None
            ax.plot(theta, r, color=color, linewidth=linewidth, alpha=1, label=current_label, linestyle=linestyle, marker=marker, markersize=1.5)
    
            if not label_added: 
                label_added=True
def plot_sum_field(Theta, Z, interp_stack, image_outfile,title='Summed Collision Field', add_colorbar=True,TV=False, all_fascicles=None, collision_field='RV_Collision', append=False, fig=None, ax=None, thresh=10, CS=False,full_circle=False,radius_title=False, device='Micra', add_legend=True, add_CS_wedge=False, LV_circle=True,horiz_colorbar=True): 
    
    sum_field = np.nansum(interp_stack, axis=0)
    
    print(f"sum field = {sum_field}")
    #print(interp_stack.shape[0], mean_field.shape, std_field.shape)
    #print(np.min(mean_field), np.max(mean_field), np.min(std_field), np.max(std_field), np.min(sum_field), np.max(sum_field))

    numcases = interp_stack.shape[0]
    # Normalize summed field to [0, 1] for coloring
    print(f"numcases={numcases}")
    sum_field_normalized = 100*sum_field / numcases
    print(f'Summed field normalized: min={np.min(sum_field_normalized)}, max={np.max(sum_field_normalized)}')
    
    # Plot the sum field
    if not append or fig is None or ax is None: 
        fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 1)
    plot_fascicles(ax, all_fascicles)
    cmap, bounds, norm, contour_colour, under_color=create_collision_colormap(collision_field, n_levels=10)

    if not append: 
        # Use pcolormesh with masking
        c = ax.pcolormesh(Theta, Z, sum_field_normalized,
                        cmap=cmap, norm=norm, shading='auto', alpha=1) #,
                        #vmin=0, vmax=100)  # vmin=1 ensures 0s appear as black

        # Use the same bounds as your normalization for consistent contours
        contour_levels = bounds # [1::]  # Every other boundary for cleaner look
        contours = ax.contour(Theta, Z, sum_field_normalized, 
                            levels=contour_levels, colors=contour_colour, #cmap=cmap, #
                            linewidths=0.6, alpha=0.8)
    else: 
        #masked_field = np.ma.masked_where(sum_field_normalized <= 10, sum_field_normalized)
        #c = ax.pcolormesh(Theta, Z, masked_field,
        #                 norm=norm, shading='auto',cmap=cmap) #, alpha=0.7) #, alpha=0.5
        
        thresh=thresh-1
        # Mask values <= 10
        masked_field = np.ma.masked_where(sum_field_normalized <= thresh, sum_field_normalized)

        # Copy cmap so we don't overwrite the original
        cmap_mod = cmap.copy()

        # Get RGBA array
        colors = cmap_mod(np.linspace(0, 1, cmap_mod.N))  # shape (N, 4)

        # Define white RGBA and under_color RGBA
        white_rgba = mpl.colors.to_rgba("#ffffff", alpha=0.0)
        under_rgba = mpl.colors.to_rgba(under_color, alpha=1.0)
        #print(colors)
        # Change indices 0–5 to white
        #colors[0] = white_rgba

        # Change indices 6–9 to under_color
        colors[0:10] = under_rgba  # adjust slice length to match colormap size
        #print(colors)
        # Create new colormap
        cmap_mod = mpl.colors.ListedColormap(colors)

        # Optionally set masked values to under_color
        #cmap_mod.set_bad(color=white_rgba)
        #uc = mpl.colors.to_rgba(under_color, alpha=0.3)  # under_color with transparency
        #ax.set_facecolor(uc)  # This sets the background to that transparent color

        # Plot
        c = ax.pcolormesh(
            Theta, Z, masked_field,
            norm=norm, shading='auto', cmap=cmap_mod, alpha=0.3
        )

        # Use the same bounds as your normalization for consistent contours
        #print(bounds[1::4])
        contour_levels = [thresh] #bounds # [::2]  # Every other boundary for cleaner look
        contours = ax.contour(Theta, Z, sum_field_normalized, 
                            levels=contour_levels, colors=contour_colour, #
                            linewidths=2)
        # clear legend if we are appending 
        #plt.legend([], [])
    # Plot each fascicle set as its own colored line

    # Add colorbar
    if add_colorbar:
        if radius_title: 
            fig.colorbar(c, ax=ax, label='Percentage of cases with collision', 
                orientation='vertical', 
                location='left',
                shrink=1,        # Make it 40% of the width
                pad=0,           # Distance from the plot
                aspect=20)       
        elif horiz_colorbar: # This is the default option
                fig.colorbar(c, ax=ax, label='Percentage of cases with collision', 
                orientation='horizontal', 
                location='bottom',
                shrink=0.4,        # Make it 40% of the width
                pad=0.1,           # Distance from the plot
                aspect=20)       
        else: 
                fig.colorbar(c, ax=ax, label='Percentage of cases with collision', 
                orientation='vertical', 
                location='right',
                shrink=1,        # Make it 40% of the width
                pad=0,           # Distance from the plot
                aspect=20)       
        #handles, labels = ax.get_legend_handles_labels()
        #unique = dict(zip(labels, handles))
        #ax.legend(unique.values(), unique.keys())

    # if all_TV_ring_polar_coords ==[]: 
    #     print('No TV annulus')
    # else:         
    #     #print(all_TV_ring_polar_coords)
    #     #all_TV_ring_polar_coords = np.vstack(all_TV_ring_polar_coords)
    #     #ax.scatter(np.radians(all_TV_ring_polar_coords[:,0]), all_TV_ring_polar_coords[:,1], color='#0d8975', s=1, alpha=0.2) #ce614d  #0d8975
    #     for i, annulus in enumerate(all_TV_ring_polar_coords):
    #         # Convert to radians if data is in degrees
    #         theta = np.radians(annulus[:, 0])
    #         r = annulus[:, 1]
    #         label = '20mm below \n TV annulus' if i == 0 else "_nolegend_"
    #         ax.plot(theta, r, color="#ce674d", linewidth=1.5, alpha=0.7, label=label)  # line instead of scatter

        #plt.legend()    
        #ax.legend(loc='upper right')
    
    # get unique labels
    #handles, labels = plt.gca().get_legend_handles_labels()
    #by_label = dict(zip(labels, handles))  # removes duplicates

    #plt.legend(by_label.values(), by_label.keys())

    ax.set_title(title, pad=20, fontsize=14)
    add_AHA_lines(ax, full_circle)
    
    if not full_circle: 
        ax.set_thetamin(210)
        ax.set_thetamax(330)
    if LV_circle:        
        # Fill the gray region: 330–360 and 0–210
        start_angle = 330
        end_angle = 210
        width = (360 - start_angle + end_angle) % 360
        ax.bar(
            np.deg2rad(start_angle),   # starting angle
            1.0,                       # radius (unit circle)
            width=np.deg2rad(width),   # angular width
            bottom=0.0,                # from center out
            color="#555555",
            alpha=0.3,
            align="edge"
        )
    plt.tight_layout()
    AHA_mapping_regions=get_AHA_config()

    if add_CS_wedge: 
        ax=add_region_wedge(ax)

    # Save the figure          
    if append: 
        return fig, ax
    else:   
        if ( CS or TV ) and device=='AveirAR' and add_legend: 
            #add_CS_legend(labels=["LB", "LV sept", "LV post", "LV ant"])    
            add_CS_legend(all_fascicles) #)
            #ax.legend(loc='upper right')  
        print(f'Saving summed RV free wall polar plot to {image_outfile}')
        plt.savefig(image_outfile, dpi=300, bbox_inches='tight')
        plt.close(fig)  
        
# Method 1: 2D Delaunay Triangulation (if septum points lie roughly in a plane)
def create_septum_triangulation_2d(refmesh, septum_indices):
    """
    Create triangular mesh using 2D Delaunay triangulation
    Assumes septum points can be projected onto a 2D plane
    Preserves all point field data from original mesh
    """
    septum_indices = np.asarray(septum_indices, dtype=int)

    # Extract points
    septum_points = refmesh.points[septum_indices]
    
    # Project points onto best-fit plane or use first two principal components
    from sklearn.decomposition import PCA
    
    pca = PCA(n_components=2)
    points_2d = pca.fit_transform(septum_points)
    
    # Perform Delaunay triangulation in 2D
    tri = Delaunay(points_2d)
    
    # Create new mesh with original 3D points but new connectivity
    faces = []
    for simplex in tri.simplices:
        # PyVista format: [3, point1, point2, point3] for triangles
        faces.extend([3, simplex[0], simplex[1], simplex[2]])
    
    septum_mesh = pv.PolyData(septum_points, faces)
    
    # Copy all point data fields from original mesh
    for field_name in refmesh.point_data.keys():
        septum_mesh.point_data[field_name] = refmesh.point_data[field_name][septum_indices]
    
    # Copy any field data (metadata)
    for field_name in refmesh.field_data.keys():
        septum_mesh.field_data[field_name] = refmesh.field_data[field_name]
    
    return septum_mesh


def save_plots_per_case(points, field, lm_points, title, image_outfile, network_path, points_array, TV_ringfile=None): 
    print(f'title: {title}')
    print(f'image_outfile: {image_outfile}')
    print(f'network_path: {network_path}')
    
    fig, ax = plt.subplots(subplot_kw={'projection': 'polar'})
    fig, ax = plot_mesh_interpolate(fig, ax, polar_points=points, field=field, lm_points=lm_points, title=title)       

    # Plot the fascicles
    if network_path is not None: 
        HB_points, LB_points, RB_points, LV_fascicles_points, RV_fascicles_points = extract_fascicles(network_path, points_array)       
        ax.scatter(np.radians(HB_points[:, 0]), HB_points[:, 1], color='black', label='HB', s=10)
        ax.scatter(np.radians(LB_points[:, 0]), LB_points[:, 1], color='#32CD32', label='LB', s=10)
        ax.scatter(np.radians(RB_points[:, 0]), RB_points[:, 1], color='#E0E0E0', label='RB', s=10)
        ax.scatter(np.radians(LV_fascicles_points[:, 0]), LV_fascicles_points[:, 1], color='cyan', label='LV Fascicles', s=10)
        ax.scatter(np.radians(RV_fascicles_points[:, 0]), RV_fascicles_points[:, 1], color='yellow', label='RV Fascicles', s=10)
        ax.legend()

    if TV_ringfile is not None: 
        TV_ring=extract_TV_ring(TV_ringfile, points_array)
        #ax.scatter(np.radians(TV_ring[:,0]), TV_ring[:,1], color='#0d8975', s=10, alpha=0.1) #ce614d  #0d8975


    plt.savefig(image_outfile, dpi=300, bbox_inches='tight')
    plt.close(fig)  
    return 

def get_results_output(device, length, collision_fields=['Combined_Collision'], plot_per_case=False, plot_all_cases=True,TPM=False,CS=False,TV=False,append=False,thresh=10,full_circle=False, radius_title=False, device_title=True, add_len_in_title=False, add_legend=True,add_CS_wedge=False, screenshot_name=None, horiz_colorbar=True,conestr='', meshsize=None): 
    
    
    #print('running get_results_output(device, length,plot_per_case=False, plot_all_cases=True)')
    #all_interp_fields_RV_freewall = []
    verbose=False
    if add_len_in_title: 
        length_str=f'\n L:{length}mm'
    else: 
        length_str=''
    if device_title: 
        if device=='Micra': 
            device_str=f'Micra TPS{length_str}'
        elif device=='AveirAR': 
            device_str=f'Aveir AR{length_str}'
        elif device=='AveirVR': 
            device_str=f'Aveir VR{length_str}'
        else: 
            device_str=device
    else: 
        device_str=''

    all_TV_ring=[]
    all_HB_points=[]
    all_LB_points=[]
    all_RB_points=[]
    all_LV_fascicles_points=[]
    all_RV_fascicles_points=[]
    all_LV153=[]
    all_LV154=[]
    all_LV155=[]
    fig=None
    ax=None
    if TPM: # Transvenous PM 
        PM_str='T-PM/'
    else: # Transcatheter PM
        PM_str=''
    for c_ind, collision_field in enumerate(collision_fields): 
        all_interp_fields_RV_septum = []
        if full_circle and c_ind==0: 
            LV_circle=True
        else: 
            LV_circle=False
    

        for casenum in [1]: #range(1,14): #[1:13]:
            if casenum in [3, 5, 12]: # Skip cases with no data
                print(f'Skipping case {casenum:02d} due to no data.')
                continue
            #if plot_per_case:
                #if casenum != 11: # Only test for the case 11
                
                #print(f'Skipping case {casenum:02d} for per-case plotting.')
                #continue
            mainpath=f'/data_fast/Work/MS_HF_cohort/data/{casenum:02d}'
            segpath=f'{mainpath}/1_seg/'
            xml_file = f'{mainpath}/1_seg/ap_mv_2rv_Siemens.mps'
            if meshsize is not None: 
                init_mesh=f'{mainpath}/1_seg/RV_dec_filtered{meshsize}.vtk'
            else: 
                init_mesh=f'{mainpath}/1_seg/RV_dec_filtered.vtk'
            
            simfile=f'{mainpath}/2_motion/{device}{conestr}/L{length}/RV_collision-all_times.vtk'
            init_mesh=simfile # Use the sim file as the reference mesh to ensure same number of points
            if TV: 
                TV_ringfile=f'{mainpath}/1_seg/TV_annulus-20.vtk' # or None
            else: 
                TV_ringfile=None
            if CS: 
                network_path=f'{mainpath}/1_seg/final_network.vtk'
            else: 
                network_path=None
            
            
            # Set file paths
            #, RV_Collision, TV_Collision, 'PM_Collision', 'Combined_Collision'
            if collision_field == 'Combined_Collision':
                images_outpath=f'/data_fast/Work/MS_HF_cohort/images/{device}{conestr}/'
                add_colorbar=True
            else: 
                images_outpath=f'/data_fast/Work/MS_HF_cohort/images/{device}{conestr}/{collision_field}/'
                add_colorbar=True
            if os.path.exists(images_outpath) is False:
                print(f'Creating directory {images_outpath}')
                os.makedirs(images_outpath)
            if os.path.exists(f'{images_outpath}/case_plots') is False:
                print(f'Creating directory sub directories of {images_outpath}')
                os.makedirs(f'{images_outpath}/case_plots')
                os.makedirs(f'{images_outpath}/sum_plots')
                os.makedirs(f'{images_outpath}/case_mesh')
            if os.path.exists(f'{images_outpath}/csv') is False:
                os.makedirs(f'{images_outpath}/csv')


            # Example usage
            try:
                points_array = read_xml_point_file(xml_file)
            except FileNotFoundError:
                print(f"XML file not found for case {casenum:02d}. Skipping this case.")
                continue    
            if verbose: 
                print(f'Processing case {casenum:02d} with mesh {init_mesh}')
            refmesh=pv.read(init_mesh)
            
            # split pyvista mesh into septum and RV freewall (elemTag 0=RV free wall, 3=septum) #
            #- by dilating the LV wall 1_3_3_3 and then using vtk_scar_map to map RV_dec_filtered.vtk

            rv_freewall_indices, septum_indices = split_mesh_by_elemTag(refmesh, tag_name='elemTag',onpts=True)
            septum_mesh = refmesh.extract_points(septum_indices, adjacent_cells=False)
            
            if verbose: 
                print(f'Save files for case {casenum:02d} to {segpath}')

            # Convert the points to polar coordinates for the RV free wall and septum meshes
            septum_points = convert_to_polar_2d(septum_mesh.points, points_array)
            
            # add in the points from the xml file 
            lm_points= convert_to_polar_2d(points_array, points_array)

            # Get the TV annulus points 
            if TV_ringfile is not None: 
                TV_annulus=extract_TV_ring(TV_ringfile, points_array)
                all_TV_ring.append(TV_annulus)
                #print(TV_annulus)

            if network_path is not None: 
                HB_points, LB_points, RB_points, LV_fascicles_points, RV_fascicles_points,LV153,LV154,LV155 = extract_fascicles(network_path, points_array)
                #all_HB_points.append(HB_points)
                all_LB_points.append(LB_points)
                #all_RB_points.append(RB_points)
                all_LV153.append(LV153)
                all_LV154.append(LV154)
                all_LV155.append(LV155)
                #all_LV_fascicles_points.append(LV_fascicles_points)
                #all_RV_fascicles_points.append(RV_fascicles_points)

            # Read in mesh 
            mesh=pv.read(simfile)
            # check number of points in mesh 
            if mesh.n_points != refmesh.n_points:
                raise ValueError(f"Mesh point count mismatch for case {casenum:02d}: "
                                 f"expected {refmesh.n_points}, got {mesh.n_points}")
            #septum_field=mesh.extract_points(septum_indices, adjacent_cells=False).point_data[collision_field]

            # Try / except loop to check for field names - if only {collision_field}_Percent exists, then /10 to get {collision_field}
            try: 
                septum_field_name = collision_field
                percent_field_name = f"{collision_field}_Percent"

                septum_pts = mesh.extract_points(septum_indices, adjacent_cells=False)
                pd = septum_pts.point_data
                # Case 1: collision_field exists
                if septum_field_name in pd:
                    septum_field = pd[septum_field_name]
                # Case 2: collision_field_Percent exists → create collision_field
                elif percent_field_name in pd:
                    # divisor = 100.0 if collision_field.endswith("_Cone") else 10.0
                    #pd[septum_field_name] = pd[percent_field_name] / divisor
                    # pd[septum_field_name] = pd[percent_field_name]
                    # septum_field = pd[septum_field_name]
                    values = pd[percent_field_name]     
                    # We want to check if any timepoint has collision, so if has a value - set to 1, otherwise if 0 set to 0
                    septum_field = np.where(values != 0, 1, 0)
                # Case 3: neither field exists → error
                else:
                    raise KeyError(
                        f"Neither '{septum_field_name}' nor '{percent_field_name}' "
                        f"exists in point_data. Available fields: {list(pd.keys())}"
                    )

            except Exception as e:
                raise RuntimeError(f"Failed to load collision field: {e}")


            submesh = create_septum_triangulation_2d(mesh,septum_indices)
            submesh.save(f"{segpath}/septum_submesh.vtk")
            # Interpolate the field onto a regular grid in polar coordinates
            
            if plot_all_cases: 
                Theta, Z, interp = interpolate_field(septum_points, septum_field)
                all_interp_fields_RV_septum.append(interp)
            # Plot for each case
            # Plot with the RV septum points
            if plot_per_case:            
                image_outfile=f'{images_outpath}/case_plots/RV_septum_{device}_L{length}_case{casenum:02d}.png'
                print(f'Plotting RV septum in polar coordinates for case {casenum:02d} to {image_outfile}')            
                title=f'{device} L{length}: \ncase{casenum:02d} RV septum' 
                
                save_plots_per_case(septum_points, septum_field, lm_points, title, image_outfile, network_path, points_array,TV_ringfile=TV_ringfile)
                # Plot with the septum points
                
        if plot_all_cases: 
            # Stack and compute sum across cases then find the mean/std in each AHA region 
            interp_stack = np.stack(all_interp_fields_RV_septum, axis=0)
            sum_field = np.sum(interp_stack, axis=0)  # shape: (200, 180)


            region_stats=[]
            AHA_mapping_regions=get_AHA_config()
            for AHA_region in [2,3,8,9,14,20,21,22,23]: 
                theta_range=AHA_mapping_regions[AHA_region]['theta_range']
                z_range=AHA_mapping_regions[AHA_region]['z_range']
                
                theta_grid = np.linspace(210, 330, 180)
                z_grid = np.linspace(0, 1, 200)
                
                region=extract_region_from_grid(sum_field, theta_grid, z_grid, theta_range, z_range)
                # Save region values to csv
                if verbose: 
                    print(f'Extracted region {AHA_region} with shape {region.shape}')
                csv_region_outfile=f'{images_outpath}/csv/{AHA_region}_L{length}.csv'
                with open(csv_region_outfile, 'w', newline='') as csvfile:
                    writer = csv.writer(csvfile)
                    writer.writerow(['theta', 'z', 'collision_value'])
                    for i in range(region.shape[0]):
                        for j in range(region.shape[1]):
                            writer.writerow([theta_grid[j], z_grid[i], region[i, j]])
                
                region_mean=np.mean(region)*10 # To get the percentage /10cases * 100 
                region_std=np.std(region)*10
                print(f'AHA region {AHA_region}: Mean % of collisions: {np.mean(region)*10:1.1f}%')
                if verbose:                     
                    print(region)
                region_stats.append([AHA_region, region_mean, region_std])
            if screenshot_name is None: 
                image_outfile=f'{images_outpath}/sum_plots/RV_septum_polar_sum_{device}_L{length}.png'
            else: 
                image_outfile=screenshot_name
            csv_outfile=f'{images_outpath}/sum_plots/RV_septum_polar_sum_{device}_L{length}.csv'
            print(csv_outfile)

            fascicles_props, gray_base_props = get_fascicle_config()
            all_fascicles=set_all_fascicles(fascicles_props, all_HB_points, all_LB_points, all_RB_points, 
                                all_LV_fascicles_points, all_RV_fascicles_points,all_LV153, all_LV154, all_LV155, all_TV_ring)
            
            if append: 
                if radius_title: 
                    diameter=float(get_diameter(device, length))
                    title=f"{device_str} D:{diameter:1.1f}mm"
                else: 
                    title=f'{device_str}'#  \nCombined Collision: \n>{thresh}% Collision Risk'
                    
                fig,ax=plot_sum_field(Theta, Z, interp_stack, image_outfile,title=title, add_colorbar=False,TV=TV, 
                        all_fascicles=all_fascicles,collision_field=collision_field, append=append, fig=fig, ax=ax, thresh=thresh, full_circle=full_circle, device=device, add_legend=add_legend,add_CS_wedge=add_CS_wedge, LV_circle=LV_circle)

            else:
                if radius_title: 
                    diameter=float(get_diameter(device, length))
                    title=f"{device_str} D:{diameter:1.1f}mm"
                else: 
                    title= f"{device_str} {collision_field.replace('_', ' ')}"

                plot_sum_field(Theta, Z, interp_stack, image_outfile,title=title, add_colorbar=add_colorbar,TV=TV, 
                        all_fascicles=all_fascicles,collision_field=collision_field, append=append, CS=CS, full_circle=full_circle, radius_title=radius_title, device=device,add_legend=add_legend,add_CS_wedge=add_CS_wedge,LV_circle=LV_circle, horiz_colorbar=horiz_colorbar)
            

            # Write to CSV            
            with open(csv_outfile, 'w', newline='') as csvfile:
                writer = csv.writer(csvfile)
                writer.writerow(['AHA_region', 'mean', 'std'])
                for row in region_stats:
                    AHA_region, mean_val, std_val = row
                    writer.writerow([AHA_region, f"{mean_val:.4f}", f"{std_val:.4f}"])



    if plot_all_cases and append: 
        if ( CS or TV ) and device=='AveirAR' and add_legend: 
            add_CS_legend(all_fascicles) #labels=["LB", "LV sept", "LV post", "LV ant"])
            print('add CS legend')
            #ax.legend(loc='upper right')
        #########################################################################
        images_outpath=f'/data_fast/Work/MS_HF_cohort/images/{device}{conestr}/'
        if screenshot_name is None: 
            if full_circle: 
                image_outfile=f'{images_outpath}/sum_plots/Overlay_{thresh}_{device}_L{length}_full_AHA.png'    
            else: 
                image_outfile=f'{images_outpath}/sum_plots/Overlay_{thresh}_{device}_L{length}.png'
        else: 
            image_outfile=screenshot_name
        print(f'Saving summed RV free wall polar plot to {image_outfile}')

        fig.savefig(image_outfile, dpi=300, bbox_inches='tight')
        plt.close(fig)      



In [3]:
def fig1e(): 
    length=15 
    thresh=110    
    get_results_output('AveirAR', length, collision_fields=['RV_Collision'], plot_per_case=False, plot_all_cases=True,TPM=False,TV=False,CS=True, append=True, thresh=thresh, full_circle=True, device_title=False, add_len_in_title=False, radius_title=False, add_legend=False,add_CS_wedge=True)
def fig5(): 
    length=15 
    thresh=110    
    get_results_output('AveirAR', length, collision_fields=['RV_Collision'], plot_per_case=False, plot_all_cases=True,TPM=False,TV=False,CS=True, append=True, thresh=thresh, full_circle=False, device_title=False, add_len_in_title=False, radius_title=False, add_legend=False,add_CS_wedge=True)
def fig2():

    TPM=False # For leadless PM simulation
    #TPM=True # For transvenous PM simulation
    # Map to AHA
    # AveirVR 38mm, AveirAR 32.2mm, Micra 25.9mm
    for device in ['AveirAR']: #, 'AveirVR']: # , 'AveirAR', 'AveirVR']: #, 'AveirAR']: #, 'Micra']: #
        if device == 'AveirVR':
            init_len= 38
        elif device == 'AveirAR': 
            init_len= 32.2
        elif device == 'Micra': 
            init_len = 25.9
        else: 
            print(f'WARNING - DEVICE: {device} is not AveirVR, AveirAR or Micra')        

            #, RV_Collision, TV_Collision, 'PM_Collision', 'Combined_Collision'
        
        for length in [init_len]: #[15, 20, 25, 30, 35, 40, 45]:  #[22, 27, 32, 37, 42]: init_len, [init_len]: #, 
            
            # TV fig 2 A B C
            for collision_field in ['RV_Collision','PM_Collision' ]:  #, 'PM_Collision']: 
                get_results_output(device, length,collision_fields=[collision_field], plot_per_case=False, plot_all_cases=True,TPM=TPM,TV=False,CS=False, append=False, radius_title=False, device_title=False, add_legend=False, add_len_in_title=False, add_CS_wedge=False)
            for collision_field in ['TV_Collision']:  #, 'PM_Collision']: 
                get_results_output(device, length,collision_fields=[collision_field], plot_per_case=False, plot_all_cases=True,TPM=TPM,TV=True,CS=False, append=False, radius_title=False, device_title=False, add_legend=False, add_len_in_title=False, add_CS_wedge=False)
            for thresh in [50]:
                get_results_output(device, length, collision_fields=['RV_Collision', 'TV_Collision','PM_Collision'], plot_per_case=False, plot_all_cases=True,TPM=TPM,TV=True,CS=False, append=True, thresh=thresh, full_circle=False, device_title=True, add_len_in_title=True, radius_title=True, add_CS_wedge=False)
            #for thresh in [110]:
            #    get_results_output(device, length, collision_fields=['RV_Collision'], plot_per_case=False, plot_all_cases=True,TPM=TPM,TV=True,CS=True, append=True, #thresh=thresh, full_circle=False, device_title=True, add_len_in_title=True, radius_title=True)
def fig3(): 
    for device in ['AveirAR']: #, 'AveirVR']: # , 'AveirAR', 'AveirVR']: #, 'AveirAR']: #, 'Micra']: #
        if device == 'AveirVR':
            init_len= 38
        elif device == 'AveirAR': 
            init_len= 32.2
        elif device == 'Micra': 
            init_len = 25.9
        else: 
            print(f'WARNING - DEVICE: {device} is not AveirVR, AveirAR or Micra')        

            #, RV_Collision, TV_Collision, 'PM_Collision', 'Combined_Collision'
        
        for length in [init_len]: #[15, 20, 25, 30, 35, 40, 45]:  #[22, 27, 32, 37, 42]: init_len, [init_len]: #, 
            
            for thresh in [50]:
                if length==45: 
                    add_legend=True
                else: 
                    add_legend=False
                print(add_legend)
                get_results_output(device, length, collision_fields=['TV_Collision','RV_Collision','PM_Collision'], plot_per_case=False, plot_all_cases=True,TV=True,CS=True, append=True, thresh=thresh, full_circle=False, radius_title=True, add_legend=add_legend, add_len_in_title=True)
        
def fig4(): 
    for device in ['AveirAR']: #, 'AveirVR']: # , 'AveirAR', 'AveirVR']: #, 'AveirAR']: #, 'Micra']: #
        if device == 'AveirVR':
            init_len= 38
        elif device == 'AveirAR': 
            init_len= 32.2
        elif device == 'Micra': 
            init_len = 25.9
        else: 
            print(f'WARNING - DEVICE: {device} is not AveirVR, AveirAR or Micra')        

            #, RV_Collision, TV_Collision, 'PM_Collision', 'Combined_Collision'
        
        for length in [15, 20, 25, 30, 35, 40, 45]:  #[22, 27, 32, 37, 42]: init_len, [init_len]: #, 
            
            for thresh in [50]:
                if length==45: 
                    add_legend=True
                else: 
                    add_legend=False
                add_legend=False
                get_results_output(device, length, collision_fields=['TV_Collision','RV_Collision','PM_Collision'], plot_per_case=False, plot_all_cases=True,TV=True,CS=False, append=True, thresh=thresh, full_circle=False, radius_title=True, add_legend=add_legend, add_len_in_title=True, add_CS_wedge=False)
        
                      
def fig1d(): 
    device ='AveirAR'
    length= 32.2
    thresh=50
    get_results_output(device, length, collision_fields=['TV_Collision','RV_Collision','PM_Collision'], plot_per_case=False, plot_all_cases=True,TV=False,CS=False, append=True, thresh=thresh, full_circle=True, device_title=False, add_len_in_title=False, radius_title=False, add_legend=True)

def analysis(): 
    for device in ['Micra', 'AveirAR', 'AveirVR']: 
        if device == 'AveirVR':
            init_len= 38
        elif device == 'AveirAR': 
            init_len= 32.2
        elif device == 'Micra': 
            init_len = 25.9
        else: 
            print(f'WARNING - DEVICE: {device} is not AveirVR, AveirAR or Micra')        
        for length in [init_len, 15, 20, 25, 30, 35, 40, 45]:  #[22, 27, 32, 37, 42]: init_len, [init_len]: #, 
            for collision_field in ['RV_Collision','PM_Collision','TV_Collision', 'Combined_Collision' ]:  #, 'PM_Collision']: 
                get_results_output(device, length,collision_fields=[collision_field], plot_per_case=False, plot_all_cases=True,TPM=False,TV=False,CS=False, append=False, radius_title=False, device_title=False, add_legend=False, add_len_in_title=False, add_CS_wedge=False)

def graph_ab(): 
    thresh=50
    for coll in ['RV', 'TV', 'PM']: 
        screenshot_name=f'/data_fast/Work/MS_HF_cohort/images/graph_abstract/GraphAb_{coll}2.png'
        append=True
        get_results_output('AveirAR', 32.2, collision_fields=[f'{coll}_Collision'], plot_per_case=False, plot_all_cases=True,TPM=False,TV=False,CS=False, append=append, thresh=thresh, full_circle=False, device_title=False, add_len_in_title=False, radius_title=False, add_legend=False,add_CS_wedge=False, screenshot_name=screenshot_name,horiz_colorbar=False)
    
    # Getting the CSP pacing regions on the AHA map
    #screenshot_name=f'/data_fast/Work/MS_HF_cohort/images/graph_abstract/GraphAb_B2.png'
    #thresh=110    
    #get_results_output('AveirAR', 32.2, collision_fields=['RV_Collision'], plot_per_case=False, plot_all_cases=True,TPM=False,TV=False,CS=True, append=True, thresh=thresh, #full_circle=True, device_title=False, add_len_in_title=False, radius_title=False, add_legend=False,add_CS_wedge=True, screenshot_name=screenshot_name)

if __name__ == "__main__":
    #fig1d()
    #fig1e()
    #fig2()
    #fig3()
    #fig4()
    #fig5()
#    graph_ab()
#    analysis()
    print('no analysis')   

no analysis


In [6]:
# WITH CONE ANALYSIS - Running the simulation with a X degree cone with Y samples (note [0] is the normal and the other samples are distributed around there) to see if allowing an X degree rotation of the lead alters the collision rate of the device with the intracardiac structures: coneX(degrees)_Y(samples)
def analysis(): 
    for device in ['Micra', 'AveirAR', 'AveirVR']: #'Micra',
        if device == 'AveirVR':
            init_len= 38
        elif device == 'AveirAR': 
            init_len= 32.2
        elif device == 'Micra': 
            init_len = 25.9
        else: 
            print(f'WARNING - DEVICE: {device} is not AveirVR, AveirAR or Micra')        
        for length in [15, 30, 45]: #[init_len, 15, 20, 25, 30, 35, 40, 45]:  #[22, 27, 32, 37, 42]: init_len, [init_len]: #, 
            for collision_field in ['RV_Collision_Cone','PM_Collision_Cone','TV_Collision_Cone', 'Combined_Collision_Cone' ]:  #, 'PM_Collision']: 
                degree=10
                sample=10
                for meshsize in [1.5]: #, 2, 5]:
                    conestr=f"_cone{degree}_{sample}_mesh{meshsize}"
                    get_results_output(device, length,collision_fields=[collision_field], plot_per_case=False, plot_all_cases=True,TPM=False,TV=False,CS=False, append=False, radius_title=False, device_title=False, add_legend=False, add_len_in_title=False, add_CS_wedge=False,conestr=conestr, meshsize=meshsize)
def fig1d(): 
    device ='AveirAR'
    length= 32.2
    thresh=50
    conestr='_cone10_10'
    get_results_output(device, length, collision_fields=['TV_Collision_Cone','RV_Collision_Cone','PM_Collision_Cone'], plot_per_case=False, plot_all_cases=True,TV=False,CS=False, append=True, thresh=thresh, full_circle=True, device_title=False, add_len_in_title=False, radius_title=False, add_legend=True,conestr=conestr)

def fig4(): 
    for device in ['AveirAR']: #, 'AveirVR']: # , 'AveirAR', 'AveirVR']: #, 'AveirAR']: #, 'Micra']: #
        if device == 'AveirVR':
            init_len= 38
        elif device == 'AveirAR': 
            init_len= 32.2
        elif device == 'Micra': 
            init_len = 25.9
        else: 
            print(f'WARNING - DEVICE: {device} is not AveirVR, AveirAR or Micra')        

            #, RV_Collision, TV_Collision, 'PM_Collision', 'Combined_Collision'
        
        for length in [init_len, 15, 20, 25, 30, 35, 40, 45]:  #[22, 27, 32, 37, 42]: init_len, [init_len]: #, 
            
            for thresh in [50]:
                if length==45: 
                    add_legend=True
                else: 
                    add_legend=False
                add_legend=False
                degree=5
                sample=10
                conestr=f"_cone{degree}_{sample}"
                
                get_results_output(device, length, collision_fields=['TV_Collision_Cone','RV_Collision_Cone','PM_Collision_Cone'], plot_per_case=False, plot_all_cases=True,TV=True,CS=False, append=True, thresh=thresh, full_circle=False, radius_title=True, add_legend=add_legend, add_len_in_title=True, add_CS_wedge=False,conestr=conestr)
        
def check_all_cases(): 
    device='Micra'
    collision_field ='RV_Collision_Cone'
    degree=10
    sample=10
    conestr=f"_cone{degree}_{sample}"
    for length in [30, 35]: # 40, 45]: 
        get_results_output(device, length,collision_fields=[collision_field], plot_per_case=True, conestr=conestr)
if __name__ == "__main__":
    analysis()
    #fig1d()
    #fig4()
    #check_all_cases()

Element tags and counts: {np.int32(0): np.int64(12659), np.int32(3): np.int64(3348)}


ValueError: Mesh point count mismatch for case 01: expected 8184, got 8183